## Import Libraries

In [4]:
import re
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Using IMDB Dataset

In [5]:
df = pd.read_csv('/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')

In [6]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


## Preprocessing Function

In [7]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove punctuation & numbers
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    
    # Tokenization
    words = text.split()
    
    # Remove stopwords + Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]
    
    return " ".join(words)

## Applying Preprocessing Function

In [8]:
df["preprocessed_dataset"] = df["review"].apply(preprocess_text)

In [9]:
df["preprocessed_dataset"]

0        one reviewer mentioned watching oz episode hoo...
1        wonderful little production filming technique ...
2        thought wonderful way spend time hot summer we...
3        basically family little boy jake think zombie ...
4        petter mattei love time money visually stunnin...
                               ...                        
49995    thought movie right good job creative original...
49996    bad plot bad dialogue bad acting idiotic direc...
49997    catholic taught parochial elementary school nu...
49998    going disagree previous comment side maltin on...
49999    one expects star trek movie high art fan expec...
Name: preprocessed_dataset, Length: 50000, dtype: object

In [10]:
print("Original:\n", df["review"].iloc[0])
print("\nCleaned:\n", df["preprocessed_dataset"].iloc[0])

Original:
 One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due

## Train-Test Split

In [11]:
df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

X_train, X_test, y_train, y_test = train_test_split(
    df["review"],
    df["label"],
    test_size=0.2,
    random_state=42
)

## Feature Extraction (TF-IDF)

In [12]:
vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

## Model Training

### Naive Bayes

In [13]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

nb_preds = nb_model.predict(X_test_tfidf)

### Logistic Regression

In [14]:
lr_model = LogisticRegression()
lr_model.fit(X_train_tfidf, y_train)

lr_preds = lr_model.predict(X_test_tfidf)

### Support Vector Machine

In [15]:
svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)

svm_preds = svm_model.predict(X_test_tfidf)

## Evaluation Function

In [16]:
def evaluate_model(y_true, y_pred, model_name):
    print(f"--- {model_name} ---")
    
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("\nClassification Report:\n", classification_report(y_true, y_pred))
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\n")

In [17]:
evaluate_model(y_test, nb_preds, "Naive Bayes")
evaluate_model(y_test, lr_preds, "Logistic Regression")
evaluate_model(y_test, svm_preds, "SVM")

--- Naive Bayes ---
Accuracy: 0.8635

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.89      0.87      4961
           1       0.88      0.84      0.86      5039

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000


Confusion Matrix:
[[4405  556]
 [ 809 4230]]


--- Logistic Regression ---
Accuracy: 0.8999

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.89      0.90      4961
           1       0.89      0.91      0.90      5039

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000


Confusion Matrix:
[[4401  560]
 [ 441 4598]]


--- SVM ---
Accuracy: 0.903

Classification Report:
               precision    recall  f1-score   support

          

## Visualization (Confusion Matrix Plot)

In [18]:
import seaborn as sns

def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    
    sns.heatmap(cm, annot=True, fmt="d")
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

## Model Comparison Table

In [19]:
results = {
    "Model": ["Naive Bayes", "Logistic Regression", "SVM"],
    "Accuracy": [
        accuracy_score(y_test, nb_preds),
        accuracy_score(y_test, lr_preds),
        accuracy_score(y_test, svm_preds)
    ]
}

results_df = pd.DataFrame(results)
results_df

,Model,Accuracy
0,Naive Bayes,0.8635
1,Logistic Regression,0.8999
2,SVM,0.9030


## Try Custom Input

In [20]:
def predict_text(text, model):
    vec = vectorizer.transform([text])
    prediction = model.predict(vec)
    return "Positive" if prediction[0] == 1 else "Negative"

In [49]:
print("Naive Bayes model: ", predict_text("I did not like this movie", nb_model)) 
print("Logistic Regression model: ", predict_text("I did not like this movie", lr_model)) 
print("Support Vector Machine model: ", predict_text("I did not like this movie", svm_model)) 

Naive Bayes model:  Negative
Logistic Regression model:  Negative
Support Vector Machine model:  Negative


In [50]:
print("Naive Bayes model: ", predict_text("I enjoyed this movie", nb_model)) 
print("Logistic Regression model: ", predict_text("I enjoyed this movie", lr_model)) 
print("Support Vector Machine model: ", predict_text("I enjoyed this movie", svm_model)) 

Naive Bayes model:  Positive
Logistic Regression model:  Positive
Support Vector Machine model:  Positive


## Key Concepts

Why TF-IDF works

- Captures importance of words

Model Differences

- Naive Bayes → Fast, works well for text
  
- Logistic Regression → Strong baseline

- SVM → Powerful but slower